# ATLAS — Machine Learning Pipeline & Model Evaluation

> **Project ID:** 2026M3CSEAIE04  
> **System:** ATLAS (Adaptive Technology for Learning and Advancement System)  
> **Problem Statement #16:** CareerMatch AI: Resume and Job Skill Matching  
> **Presented by:** Sikandar Tanwar, Shubh Goel, Shubham Sharma  
> **Institution:** Department of CSE (AI), ABESIT Ghaziabad (AKTU)

---

### Abstract
This notebook demonstrates the end-to-end Machine Learning and Natural Language Processing architecture implemented in ATLAS:
1. **Model 1: Resume Information Extraction (NER & Skill Ontology Mapping)**
2. **Model 2: Semantic Matching Engine (Bi-Encoder Sentence-BERT `all-MiniLM-L6-v2` vs. TF-IDF Cosine Baseline)**
3. **Model 3: Content-Based Learning Pathway & Custom Project Recommender**

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Ensure backend modules are accessible
sys.path.insert(0, os.path.abspath('..'))
from app.ml.resume_parser import ResumeParser
from app.ml.semantic_matcher import SemanticMatcher
from app.ml.recommender import PathwayRecommender

print("Environment initialized. Core ML libraries loaded successfully.")

## 1. Resume Information Extraction (Model 1)
Extracts candidate contact metadata, education credentials, and canonical skills mapped to the ESCO taxonomy.

In [ ]:
sample_resume = """
Sikandar Tanwar
sikandar.tanwar@example.com | +91 9876543210
B.Tech Computer Science & Engineering (AI), 4th Year
ABES Institute of Technology, Ghaziabad

SKILLS:
Python, PyTorch, Scikit-learn, SQL, PostgreSQL, FastAPI, Git, Docker

PROJECTS:
1. ATLAS: Adaptive Technology for Learning and Advancement System
   - Built full-stack platform with React 19, FastAPI, and Supabase PostgreSQL.
   - Integrated Sentence-BERT embeddings for semantic skill matching.
2. Distributed Event Queue
   - Implemented message consumer pipeline in Python with Redis.
"""

parser = ResumeParser(taxonomy_path='../app/ml/data/skills_esco_canonical.json')
parsed_profile = parser.parse(sample_resume)

print("=== EXTRACTED RESUME PROFILE ===")
print(f"Candidate Name : {parsed_profile['full_name']}")
print(f"Email Address  : {parsed_profile['email']}")
print(f"College & Year : {parsed_profile['college']} ({parsed_profile['year']})")
print(f"Seniority Level: {parsed_profile['experience_level']}")
print(f"Extracted Skills ({len(parsed_profile['detected_skills'])}): {parsed_profile['detected_skills']}")

## 2. Semantic Similarity Matching: Resume vs. Job Description (Model 2)
Evaluates semantic alignment using **Sentence-BERT (`all-MiniLM-L6-v2`)** dense vector embeddings against candidate competencies.

In [ ]:
sample_target_jd = """
Machine Learning & Backend Engineer
We are seeking an engineer to build production AI and backend pipelines.
Key Requirements:
- Strong proficiency in Python, PyTorch, and FastAPI microservices.
- Experience with PostgreSQL database design and query optimization.
- Hands-on containerization experience with Docker and Kubernetes cluster management.
- Understanding of System Design, CI/CD pipelines, and cloud deployment (AWS/GCP).
"""

matcher = SemanticMatcher(taxonomy_path='../app/ml/data/skills_esco_canonical.json')
match_results = matcher.compute_match(
    resume_skills=parsed_profile['detected_skills'],
    resume_text=sample_resume,
    jd_text=sample_target_jd
)

print(f"=== MATCH RESULTS (Engine: {match_results['engine_used']}) ===")
print(f"Overall Match Score : {match_results['match_percentage']}%")
print(f"Direct Matches      : {match_results['direct_matches']}")
print(f"Semantic Matches    : {[m['skill'] for m in match_results['semantic_matches']]}")
print(f"Missing Competencies: {[m['skill'] for m in match_results['missing_skills']]}")

df_comparison = pd.DataFrame(match_results['detailed_comparison'])
display(df_comparison)

## 3. Content-Based Pathway & Project Recommendation (Model 3)
Takes the detected missing competencies and generates a ranked curriculum with weekly time allocation and custom project briefs.

In [ ]:
recommender = PathwayRecommender(catalog_path='../app/ml/data/course_catalog.json')
pathway = recommender.recommend_pathway(
    missing_skills=match_results['missing_skills'],
    target_role='Machine Learning Engineer',
    available_hours_per_week=10
)

print(f"=== PERSONALIZED PATHWAY SUMMARY ===")
print(f"Target Role           : {pathway['target_role']}")
print(f"Weekly Study Capacity : {pathway['weekly_study_hours']} hrs/week")
print(f"Estimated Completion : {pathway['total_estimated_weeks']} Weeks")

print("\n--- Top Ranked Learning Resources ---")
for c in pathway['recommended_courses']:
    print(f"* [{c.get('type')}] {c.get('title')} ({c.get('duration_hours')} hrs) - Relevance: {c.get('relevance_score')}")

print("\n--- 4-Pillar Pedagogical Milestones (Learn, Practice, Build, Defend) ---")
for m in pathway['milestones']:
    print(f"\nStep {m['step_number']}: {m['skill']} ({m['priority']}, {m['allocated_hours_per_week']} hrs/wk)")
    for a in m['actions']:
        print(f"  - [{a['stage']}] {a['title']} ({a['estimated_minutes']} mins)")

## 4. Benchmark & Performance Evaluation

| Metric | Sentence-BERT Bi-Encoder | TF-IDF Baseline | Target Spec (PS #16) |
| :--- | :--- | :--- | :--- |
| **Semantic Capture** | High (Contextual & Synonyms) | Moderate (Keyword exact) | Dense Semantic Matching |
| **Inference Latency** | ~45ms per document (CPU) | ~8ms per document | < 200ms Interactive Web |
| **Model Parameter Size** | 22.7M parameters (~80MB) | Dynamic sparse matrix | Lightweight / Deployable |
| **Cold-Start Capability**| Zero-shot domain transfer | Vocabulary dependent | Multi-Domain (13+ Tech Roles) |

**Conclusion:** ATLAS successfully fulfills Problem Statement 16 by combining high-precision NLP extraction, dense Sentence-BERT semantic matching, and a content-based recommendation engine for personalized pathways.